
# 05_Evaluation_Cheat_Sheet

## What this notebook is for
This is a demo-day cheat sheet for the **evaluation** folder.  

## Folder coverage
Files covered:
- `calibration.py`
- `eval_utils.py`
- `mcnemar.py`
- `model_evaluations.py`
- `plot_curves.py`
- `robustness_mini.py`
- `save_test_preds.py`
- `summarise_errors.py`

## One important note
The uploaded `calibration.py` content appears to match your data loading logic rather than calibration logic.  
So in this notebook, the **calibration section is flagged as needing verification** against your real project copy before demo day.

---

## 1. Big-picture evaluation story

A stronger version:

> “I did not just report a single accuracy number.  
> I evaluated the models from multiple angles: standard classification metrics, threshold behaviour, probability quality, statistical comparison, robustness, and qualitative error analysis.”

---



# 2. Metrics cheat sheet

## Core classification metrics I measured

### Accuracy
**Meaning:** the proportion of all predictions that were correct.  
**Use in my project:** a simple headline metric for overall correctness.  
**Good demo line:**  
> “Accuracy tells me the overall percentage of correct predictions, but by itself it is not enough for phishing detection because class balance and error type both matter.”

### Precision
**Meaning:** out of all emails predicted as phishing, how many were actually phishing.  
**Use in my project:** important because false alarms reduce trust in the system.  
**Good demo line:**  
> “Precision matters because if the system flags too many legitimate emails, users will stop trusting it.”

### Recall
**Meaning:** out of all actual phishing emails, how many the model successfully caught.  
**Use in my project:** important because missed phishing emails are dangerous.  
**Good demo line:**  
> “Recall matters because a false negative means a phishing email slips through.”

### F1 score
**Meaning:** combining precision and recall into one number.
It makes it useful when you want to catch phishing emails without wrongly flagging too many legitimiate emails.

**Use in my project:** a balanced single metric when both false positives and false negatives matter.  
**Good demo line:**  
> “It is a balance score between missing phishing emails and over-flagging safe emails.”

### ROC-AUC
**Meaning:** how well the model separates phishing and legitimate emails overall across all possible thresholds.

**Use in my project:** threshold-independent ranking quality.  
**Good demo line:**  
> “ROC-AUC tells me how well the model can distinguish phishing from legitimate emails overall, across all possible thresholds.”

### PR-AUC
**Meaning:** PR-AUC looks at the relationship between precision and recall across different thresholds.
Finds out how well the model catches phishing emails while still keeping flagged emails trustworthy

**Use in my project:** especially useful for phishing because positive class performance matters a lot.  
**Good demo line:**  
> “It focuses on how well the model performs on phishing detection specifically, not just overall separation.”

### Confusion matrix
**Meaning:** counts of true positives, true negatives, false positives, and false negatives.  
**Use in my project:** shows what kinds of mistakes the model makes.  
**Good demo line:**  
> “The confusion matrix lets me see whether performance is coming from genuinely catching phishing or just predicting the majority class.”

### Best threshold / threshold tuning
**Meaning:** choosing a decision threshold other than 0.5 to optimise a metric such as F1.  
**Use in my project:** compare default threshold and best-threshold predictions.  
**Good demo line:**  
> “I saved both default-threshold and tuned-threshold predictions so I could analyse performance fairly and transparently.”

---

## Additional evaluation angles I measured

### McNemar’s test
**Meaning:** statistical test to compare paired model predictions on the same test samples.  
**Use in my project:** checking whether Stage 3’s improvement over Stage 1 is statistically significant rather than just random variation.  
**Good demo line:**  
> “McNemar’s test compares Stage 1 and Stage 3 on the same test samples by looking at the cases where one gets it right and the other gets it wrong. That tells me whether the improvement is statistically meaningful, not just a small difference in score.”

### Calibration
**Meaning:** whether predicted probabilities match actual observed frequencies.  
**Use in my project:** checking whether a 0.8 phishing probability really behaves like an 80% phishing risk.  
**Good demo line:**  
> “Calibration matters because a probability score should mean something, not just rank examples.”

### Robustness / prediction flip rate
**Meaning:** how often the model changes its prediction after small realistic perturbations.  
**Use in my project:** stress testing stability.  
**Good demo line:**  
> “The lower the flip rate, the more stable the model is to mild changes in wording.”

### Qualitative error analysis
**Meaning:** looking at the model’s top false positives and false negatives.  
**Use in my project:** understanding failure patterns, not just reporting aggregate metrics.  
**Good demo line:**  
> “I also inspected what the model gets wrong, because a good evaluation is not just numbers; it is also understanding the error patterns.”

---



# 3. File-by-file cheat sheet

---

## `calibration.py`

### What it is supposed to do
Based on the filename, this should be the script for **probability calibration analysis**.

### What calibration means
Calibration checks whether the predicted probabilities are trustworthy.  
For example:
- if the model gives many emails a score around `0.80`
- then roughly 80% of those emails should actually be phishing

### Why it matters
A model can have strong ranking performance but still give poorly calibrated probabilities.  
That matters if you want to interpret the score as **risk**.

### What outcome it should produce
Typical calibration outputs are:
- calibration curves / reliability plots
- expected calibration error style summaries
- maybe bin-level comparisons between predicted and observed risk

### What to say in the demo
> “Calibration checks whether the probability scores are meaningful.  
> For example, if the model says 0.8 phishing probability, I want that to genuinely behave like high risk rather than just being an arbitrary score.”

### Important note for you
The uploaded `calibration.py` does **not** currently contain calibration logic.  
It appears to contain the same loader logic as `loader.py`, so please verify your actual project copy before relying on this section in the demo.

### Short speech topics
- probability quality
- risk interpretation
- reliability of confidence scores
- difference between ranking well and being well-calibrated

---

## `eval_utils.py`

### What it does
This file contains a small helper that:
1. ensures the `artifacts/eval` directory exists
2. saves **Stage 1 per-sample predictions** into a CSV file

### Why it exists
This is the plumbing that makes later evaluation possible.  
Without saving per-email outputs, you cannot easily:
- plot PR and ROC curves
- compare thresholds
- run statistical tests
- inspect sample-level behaviour later

### What outcome it produces
A CSV with:
- `id`
- `y_true`
- `prob`
- `pred_0_5`
- `pred_best`
- `threshold_best`

### What to say in the demo
> “This utility file prepares the evaluation folder and saves Stage 1 predictions sample by sample, which I then reuse for plotting curves and deeper analysis.”

### Short speech topics
- reproducibility
- per-sample prediction storage
- making later evaluation steps possible

---

## `mcnemar.py`

### What it does
This file compares **Stage 1** and **Stage 3** using **McNemar’s test**.

### What it checks
It does not just ask which model has higher accuracy.  
It checks, on the **same test emails**, whether Stage 3 wins enough disagreement cases for the improvement to be statistically significant.

### Key logic
It:
1. loads Stage 1 (b) and Stage 3 (c) prediction CSVs
2. checks that the files contain the same test set
3. builds the McNemar contingency counts
4. calculates:
   - continuity-corrected chi-square statistic (Numbers of disagreement with a mathematical formula)
   - approximate p-value (Outcome of the formula and estimation on the differences between stage 1 and stage 3s significance)
   - exact binomial p-value (binomial means treating the tests like a 50/50 and checks whether one model wins those disagreements much more often than expected by chance).
5. writes a text report

### What outcome it produces
A report file such as:
- `mcnemar_stage1_vs_stage3.txt`

This tells you whether the difference is statistically significant at the 5% level.

### What to say in the demo
> “McNemar’s test helped me check whether Stage 3’s improvement over Stage 1 was statistically meaningful on the same held-out test set, rather than just a small random difference.”

### Short speech topics
- paired model comparison
- same held-out split
- disagreement cases
- p-value and significance

### Backup explanation if asked
- **chi-square statistic:** a score measuring how unbalanced the disagreement cases are
- **p-value:** whether that imbalance is large enough to count as a real difference

---

## `model_evaluations.py`

### What it does
This is a lightweight helper file for building and writing prediction rows.

### Main functions
- `make_pred_row(...)`  
  Creates one prediction row as a dictionary
- `write_test_predictions_csv(...)`  
  Writes a list of prediction rows to CSV
- `timestamped_copy_path(...)`  
  Creates a timestamped archive filename

### Why it exists
It standardises the structure of saved evaluation outputs.

### What outcome it produces
Structured prediction CSV files that can later be used by plotting, reporting, or comparison scripts.

### What to say in the demo
> “This file standardises how prediction rows are created and saved, so the evaluation pipeline stays consistent across runs.”

### Short speech topics
- standardised output format
- reusable helper functions
- timestamped archiving

---

## `plot_curves.py`

### What it does
This file loads the saved prediction CSVs for Stage 1, Stage 2, and Stage 3, then plots:
- **Precision-Recall curves**
- **ROC curves**

### Why it matters
These curves show model behaviour **across all thresholds**, not just at one cutoff like 0.5.

### What outcome it produces
Saved plot images:
- `pr_curves.png`
- `roc_curves.png`

It also prints summary values such as:
- PR-AUC
- ROC-AUC

### What to say in the demo
> “This file creates the PR and ROC curves for all three stages, so I can compare the models visually and also report threshold-independent metrics like PR-AUC and ROC-AUC.”

### Short speech topics
- threshold-independent evaluation
- visual comparison across stages
- why PR-AUC matters in phishing detection

### Good extra line
> “I used different line styles so overlapping curves remained readable, which helped when the models performed similarly.”

---

## `robustness_mini.py`

### What it does
This file builds a **small robustness benchmark** from the held-out test split.

It creates:
- 10 phishing emails with mild obfuscation
- 10 legitimate emails with urgency injected

Then it scores both the **original** and **perturbed** versions using:
- Stage 1
- Stage 3

### Why it matters
It checks whether the models are stable under small realistic wording changes.

### What outcome it produces
A CSV like:
- `robustness_mini_results.csv`

It also prints:
- number of samples
- Stage 1 flip rate
- Stage 3 flip rate

### What “flip rate” means
A prediction flip means the model changed class after a mild perturbation.

### What to say in the demo
> “This file is a small robustness stress test. I slightly perturb real held-out emails and check how often each model changes its decision. Lower flip rate means more stable behaviour.”

### Short speech topics
- robustness
- adversarial-style perturbation but lightweight
- model stability under mild wording changes
- Stage 1 vs Stage 3 resilience

---

## `save_test_preds.py`

### What it does
This file saves **per-sample test predictions** into `artifacts/eval/`.

### What it stores
For each sample it can store:
- stage name
- id
- true label
- probability
- default-threshold prediction
- best-threshold prediction
- best threshold value
- optional extra columns such as component probabilities

### Why it matters
This is one of the core “evidence files” in the pipeline.  
A lot of the later evaluation depends on these saved CSVs.

### What outcome it produces
Two kinds of output:
- a latest file such as `stage3_test_preds.csv`
- an optional timestamped archive copy

### What to say in the demo
> “This file saves the detailed per-email prediction outputs that the rest of the evaluation pipeline depends on, including threshold-based predictions and optional extra model scores.”

### Short speech topics
- transparent saved predictions
- latest vs timestamped outputs
- reproducibility and audit trail

---

## `summarise_errors.py`

### What it does
This file turns raw error-analysis CSVs into compact text summaries for:
- Stage 2
- Stage 3

### Why it matters
A raw CSV can be noisy and repetitive.  
This script extracts the most useful examples:
- top false positives
- top false negatives

### Key extra feature: deduplication
It creates a **dedupe key** so repeated template-like emails collapse into one representative example.

That means if many emails differ only by:
- dates
- codes
- tracking numbers

they do not flood the summary.

### What outcome it produces
Two text files:
- `error_summary_stage2.txt`
- `error_summary_stage3.txt`

### What to say in the demo
> “This script creates short readable summaries of the most important false positives and false negatives, while deduplicating repeated template-style messages so I can focus on representative error patterns.”

### Short speech topics
- qualitative error analysis
- representative mistakes
- template deduplication
- confidence-aware sorting of errors

---



# 4. Recommended demo flow for the evaluation folder

A simple order to talk through it:

1. **Start broad**
   - “The evaluation folder turns raw predictions into evidence.”
   - “It stores predictions, computes metrics, plots curves, compares models statistically, tests robustness, and summarises mistakes.”

2. **Talk about saved predictions first**
   - `save_test_preds.py`
   - `eval_utils.py`
   - `model_evaluations.py`

   Good line:
   > “I first save per-sample predictions in a consistent format, because almost every later evaluation step depends on those CSV outputs.”

3. **Move into standard metrics**
   - accuracy, precision, recall, F1
   - confusion matrix
   - PR-AUC and ROC-AUC
   - `plot_curves.py`

   Good line:
   > “I used both point metrics and full curves because a single threshold does not tell the full story.”

4. **Then move into deeper evaluation**
   - `mcnemar.py`
   - robustness
   - error summaries
   - calibration if verified

   Good line:
   > “After the standard metrics, I looked at whether the improvement was statistically significant, whether the model was stable under perturbation, and what kinds of mistakes it still made.”

5. **Finish with the big message**
   > “So the evaluation was multi-layered: not just performance, but trustworthiness, robustness, and interpretability of failure cases.”

---



# 5. Quick-fire answers for likely demo questions

## “Why not just use accuracy?”
> “Because phishing detection is sensitive to both false positives and false negatives, so I also measured precision, recall, F1, PR-AUC, ROC-AUC, and inspected the confusion matrix.”

## “Why did you save per-sample predictions?”
> “Because later analysis like PR/ROC curves, McNemar’s test, threshold comparisons, and robustness checks all depend on per-sample probabilities and labels.”

## “Why PR-AUC as well as ROC-AUC?”
> “ROC-AUC shows general separability, but PR-AUC is especially informative when the positive class matters a lot, which is the case for phishing detection.”

## “What does McNemar’s test add?”
> “It checks whether Stage 3’s improvement over Stage 1 is statistically meaningful on the same held-out samples, instead of just comparing headline metrics.”

## “What does robustness mean here?”
> “I mean whether the model keeps the same decision after small realistic text perturbations. I measured that using prediction flip rate.”

## “Why inspect errors manually?”
> “Because aggregate metrics tell me how much the model is wrong, but error summaries tell me what kinds of mistakes it is making.”

## “What is calibration?”
> “Calibration is about whether the predicted probabilities are trustworthy as probabilities, not just useful for ranking.”

---



# 6. Shortest possible speech notes per file

## `calibration.py`
- checks whether probability scores are trustworthy  
- use if asked about risk interpretation

## `eval_utils.py`
- helper for creating eval folder and saving Stage 1 predictions  
- enables later plotting and comparison

## `mcnemar.py`
- statistical comparison of Stage 1 vs Stage 3  
- tests if improvement is significant on the same test set

## `model_evaluations.py`
- helper for building prediction rows and CSV outputs  
- keeps output structure consistent

## `plot_curves.py`
- plots PR and ROC curves for Stage 1, Stage 2, and Stage 3  
- gives PR-AUC and ROC-AUC

## `robustness_mini.py`
- small perturbation-based stability test  
- reports flip rate before vs after mild text changes

## `save_test_preds.py`
- saves detailed per-sample predictions  
- creates both latest and timestamped outputs

## `summarise_errors.py`
- creates readable summaries of top false positives and false negatives  
- deduplicates repeated template-like mistakes

---



# 7. Final closing line for the evaluation folder

> “Overall, the evaluation folder gave me a structured way to prove performance from multiple angles: standard metrics, curve-based analysis, saved per-sample outputs, statistical significance testing, robustness checks, and qualitative error summaries.”

A stronger alternative:

> “The key point is that I evaluated the system as a security model, not just as a classifier.  
> That means I looked at correctness, confidence, stability, significance, and failure patterns.”

---
